## YOLO

In [ ]:
import cv2
import csv
import pandas as pd
from ultralytics import YOLO

In [ ]:
VIDEO_PATH = "C0142.mp4"
OUTPUT_CSV = "detecciones.csv"
OUTPUT_VIDEO = "resultado.mp4"

# Modelos
model_general = YOLO("yolo11n.pt")
model_matriculas = YOLO("runs/matriculas_yolo11n_1280_min001/weights/best.pt")

CLASSES = [0, 1, 2, 3, 5]

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "identificador_tracking",
    "x1", "y1", "x2", "y2",
    "matricula_en_su_caso",
    "confianza_matricula",
    "mx1", "my1", "mx2", "my2",
    "texto_matricula"
]

csv_file = open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig")

writer = csv.DictWriter(csv_file, fieldnames=headers)
writer.writeheader()

frame_number = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    result_general = model_general.track(
        frame,
        persist=True,
        tracker="custom_bytetrack.yaml",
        classes=CLASSES,
        conf=0.5,
        verbose=False
    )[0]

    result_plates = model_matriculas.predict(
        frame,
        conf=0.4,
        verbose=False
    )[0]

    plates = []

    if result_plates.boxes is not None:

        for box in result_plates.boxes:

            px1, py1, px2, py2 = map(int, box.xyxy[0].tolist())

            plate_conf = float(box.conf[0])

            plates.append({
                "x1": px1,
                "y1": py1,
                "x2": px2,
                "y2": py2,
                "conf": plate_conf
            })

            cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 0, 255), 2)

            cv2.putText(frame, f"Plate {plate_conf:.2f}", (px1, max(20, py1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    boxes = result_general.boxes

    if boxes is not None:

        for i, box in enumerate(boxes):

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            confidence = float(box.conf[0])
            class_id = int(box.cls[0])

            object_type = model_general.names[class_id]

            tracking_id = None

            if boxes.id is not None:
                tracking_id = int(boxes.id[i])

            # Buscar matrícula dentro del vehículo
            matched_plate = None

            if class_id in [2, 3, 5, 7]:

                for plate in plates:

                    cx = (plate["x1"] + plate["x2"]) / 2
                    cy = (plate["y1"] + plate["y2"]) / 2

                    if (x1 <= cx <= x2 and y1 <= cy <= y2):
                        matched_plate = plate
                        break

            row = {
                "fotograma": frame_number,
                "tipo_objeto": object_type,
                "confianza": confidence,
                "identificador_tracking": tracking_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "matricula_en_su_caso": int(
                    matched_plate is not None
                ),
                "confianza_matricula": (
                    matched_plate["conf"] if matched_plate else ""
                ),
                "mx1": matched_plate["x1"] if matched_plate else "",
                "my1": matched_plate["y1"] if matched_plate else "",
                "mx2": matched_plate["x2"] if matched_plate else "",
                "my2": matched_plate["y2"] if matched_plate else "",
                "texto_matricula": ""
            }

            writer.writerow(row)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            label = f"{object_type} ID:{tracking_id}"

            cv2.putText(frame, label, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out.write(frame)

    if frame_number % 100 == 0:
        print(f"Procesados: {frame_number} fotogramas")

cap.release()
out.release()
csv_file.close()

print("Procesamiento terminado")
print("CSV generado:", OUTPUT_CSV)
print("Vídeo generado:", OUTPUT_VIDEO)


Procesados: 100 fotogramas
Procesados: 200 fotogramas
Procesados: 300 fotogramas
Procesados: 400 fotogramas
Procesados: 500 fotogramas
Procesados: 600 fotogramas
Procesados: 700 fotogramas
Procesados: 800 fotogramas
Procesados: 900 fotogramas
Procesados: 1000 fotogramas
Procesados: 1100 fotogramas
Procesados: 1200 fotogramas
Procesados: 1300 fotogramas
Procesados: 1400 fotogramas
Procesados: 1500 fotogramas
Procesados: 1600 fotogramas
Procesados: 1700 fotogramas
Procesados: 1800 fotogramas
Procesados: 1900 fotogramas
Procesados: 2000 fotogramas
Procesados: 2100 fotogramas
Procesados: 2200 fotogramas
Procesados: 2300 fotogramas
Procesados: 2400 fotogramas
Procesados: 2500 fotogramas
Procesados: 2600 fotogramas
Procesados: 2700 fotogramas
Procesados: 2800 fotogramas
Procesamiento terminado
CSV generado: detecciones.csv
Vídeo generado: resultado.mp4


In [9]:
df = pd.read_csv("detecciones.csv")

conteo = df["tipo_objeto"].value_counts()

print("Total de detecciones por clase:")
print(conteo)

Total de detecciones por clase:
tipo_objeto
car           11070
person         2517
bus             191
motorcycle       43
bicycle          35
Name: count, dtype: int64


In [10]:
conteo_unicos = (
    df[df["identificador_tracking"].notna()]
    .groupby("tipo_objeto")["identificador_tracking"]
    .nunique()
)

print("Total de objetos únicos:")
print(conteo_unicos)

Total de objetos únicos:
tipo_objeto
bicycle        1
bus            2
car           82
motorcycle     3
person        20
Name: identificador_tracking, dtype: int64


### EasyOCR

In [1]:
import cv2
import csv
import re
import easyocr
import numpy as np
from paddleocr import PaddleOCR

from ultralytics import YOLO
from collections import defaultdict, Counter

c:\Users\kilia\anaconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# EasyOCR

reader = easyocr.Reader(['es'])

def read_plate_ocr(crop):

    if crop is None or crop.size == 0:
        return "", 0.0

    # Ampliar matrícula para facilitar OCR
    #crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # Pasar a escala de grises
    #gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    # Mejorar contraste
    #gray = cv2.equalizeHist(gray)

    results = reader.readtext(crop)

    if not results:
        return "", 0.0

    # Seleccionar resultado con mayor confianza
    best = max(results, key=lambda x: x[2])

    text = best[1].replace(" ", "").upper()
    confidence = float(best[2])

    # Validar matricula: 4 números + 3 letras
    if not re.fullmatch(r"[0-9]{4}[A-Z]{3}", text):
        return "", confidence

    return text, confidence

In [ ]:
VIDEO_PATH = "C0142.mp4"
OUTPUT_VIDEO = "resultado_ocr.mp4"
OUTPUT_CSV = "detecciones_ocr.csv"

model_general = YOLO("yolo11n.pt")
model_plates = YOLO("runs/matriculas_yolo11n_1280_min001/weights/best.pt")

# OCR
reader = PaddleOCR(
    lang="en",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=False,
    cpu_threads=4
)

# Personas, bicicletas y vehículos
CLASSES = [0, 1, 2, 3, 5, 7]

# Letras permitidas en matrículas españolas
ALLOWLIST = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# Almacena las lecturas de cada vehículo
ocr_history = defaultdict(list)

# Resultado OCR más fiable por vehículo
best_plates = {}

def read_plate_ocr(crop):

    if crop is None or crop.size == 0:
        return "", 0.0

    # Ampliar matrícula
    crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # Convertir BGR a RGB
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    # Ejecutar PaddleOCR
    results = reader.predict(crop_rgb)

    if not results:
        return "", 0.0

    candidates = []

    for result in results:

        data = result.json

        if isinstance(data, dict):
            data = data.get("res", data)

        texts = data.get("rec_texts", [])
        scores = data.get("rec_scores", [])

        for text, confidence in zip(texts, scores):

            text = re.sub(r"[^A-Z0-9]", "", text.upper())

            confidence = float(confidence)

            # Validar matrícula española
            if re.fullmatch(r"[0-9]{4}[BCDFGHJKLMNPRSTVWXYZ]{3}", text):
                candidates.append((text, confidence))

    if not candidates:
        return "", 0.0

    # Seleccionar lectura de mayor confianza
    return max(candidates, key=lambda x: x[1])


def update_ocr(vehicle_id, text, confidence):

    if vehicle_id is None or not text:
        return

    # Evitar lecturas de muy baja confianza
    if confidence < 0.30:
        return

    ocr_history[vehicle_id].append((text, confidence))

    # Contabilizar lecturas
    counts = Counter(item[0] for item in ocr_history[vehicle_id])

    # Elegir la lectura más repetida
    selected_text, occurrences = counts.most_common(1)[0]

    # Confianza media de las lecturas seleccionadas
    selected_confidences = [conf for plate, conf in ocr_history[vehicle_id] if plate == selected_text]

    best_plates[vehicle_id] = {
        "text": selected_text,
        "confidence": float(np.mean(selected_confidences)),
        "occurrences": occurrences
    }

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# Cabecera csv
headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "identificador_tracking",
    "x1", "y1", "x2", "y2",
    "matricula_en_su_caso",
    "confianza_matricula",
    "mx1", "my1", "mx2", "my2",
    "texto_matricula",
    "confianza_ocr",
    "lecturas_coincidentes"
]

csv_file = open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig")

writer = csv.DictWriter(csv_file, fieldnames=headers)
writer.writeheader()

frame_number = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # Detección de personas y vehículos
    result_general = model_general.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=CLASSES,
        conf=0.5,
        verbose=False
    )[0]

    # Detección de matrículas
    result_plates = model_plates.predict(
        frame,
        conf=0.4,
        verbose=False
    )[0]

    plates = []

    if result_plates.boxes is not None:

        for box in result_plates.boxes:

            px1, py1, px2, py2 = map(int, box.xyxy[0].tolist())

            px1 = max(0, px1)
            py1 = max(0, py1)
            px2 = min(width, px2)
            py2 = min(height, py2)

            if px2 <= px1 or py2 <= py1:
                continue

            plate_conf = float(box.conf[0])

            plate_crop = frame[py1:py2, px1:px2]

            # OCR
            plate_text, ocr_conf = read_plate_ocr(plate_crop)

            plates.append({
                "x1": px1,
                "y1": py1,
                "x2": px2,
                "y2": py2,
                "conf": plate_conf,
                "text": plate_text,
                "ocr_conf": ocr_conf
            })

    boxes = result_general.boxes

    if boxes is not None:

        for i, box in enumerate(boxes):

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            confidence = float(box.conf[0])
            class_id = int(box.cls[0])

            object_type = model_general.names[class_id]

            tracking_id = None

            if boxes.id is not None:
                tracking_id = int(boxes.id[i])

            matched_plate = None

            # Asocia matrículas a vehículos
            if class_id in [2, 3, 5, 7]:

                candidates = []

                for plate in plates:

                    cx = (plate["x1"] + plate["x2"]) / 2
                    cy = (plate["y1"] + plate["y2"]) / 2

                    if (x1 <= cx <= x2 and y1 <= cy <= y2):
                        # Priorizar el vehículo más pequeño
                        candidates.append(plate)

                if candidates:
                    matched_plate = min(candidates, key=lambda p: ((x2 - x1) * (y2 - y1)))

            if matched_plate and tracking_id is not None:

                update_ocr(
                    tracking_id,
                    matched_plate["text"],
                    matched_plate["ocr_conf"]
                )

            # Matrícula más fiable acumulada
            plate_info = best_plates.get(
                tracking_id, {}
            ) if tracking_id is not None else {}

            final_text = plate_info.get("text", "")
            final_ocr_conf = plate_info.get("confidence", "")
            occurrences = plate_info.get("occurrences", 0)

            # Guarda frame en csv
            row = {
                "fotograma": frame_number,
                "tipo_objeto": object_type,
                "confianza": confidence,
                "identificador_tracking": tracking_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "matricula_en_su_caso": int(
                    matched_plate is not None
                ),
                "confianza_matricula": (
                    matched_plate["conf"] if matched_plate else ""
                ),
                "mx1": matched_plate["x1"] if matched_plate else "",
                "my1": matched_plate["y1"] if matched_plate else "",
                "mx2": matched_plate["x2"] if matched_plate else "",
                "my2": matched_plate["y2"] if matched_plate else "",
                "texto_matricula": final_text,
                "confianza_ocr": final_ocr_conf,
                "lecturas_coincidentes": occurrences
            }

            writer.writerow(row)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            label = f"{object_type} ID:{tracking_id}"

            if final_text:
                label += f" | {final_text}"

            cv2.putText(frame, label, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    for plate in plates:

        cv2.rectangle(frame, (plate["x1"], plate["y1"]), (plate["x2"], plate["y2"]), (0, 0, 255), 2)

        label = plate["text"] if plate["text"] else "Matricula"

        cv2.putText(frame, label, (plate["x1"], max(20, plate["y1"] - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Guarda el vídeo
    out.write(frame)

    if frame_number % 100 == 0:
        print(f"Procesados {frame_number} fotogramas")

cap.release()
out.release()
csv_file.close()

print("Procesamiento finalizado")
print("Vídeo:", OUTPUT_VIDEO)
print("CSV:", OUTPUT_CSV)

print("\nMatrículas reconocidas:")

for vehicle_id, info in best_plates.items():

    print(
        f"Vehículo ID {vehicle_id}: "
        f"{info['text']} "
        f"(confianza OCR: {info['confidence']:.2f}, "
        f"lecturas: {info['occurrences']})"
    )


Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kilia\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kilia\.paddlex\official_models\PP-OCRv6_medium_rec`.


Procesados 100 fotogramas
Procesados 200 fotogramas
Procesados 300 fotogramas
Procesados 400 fotogramas
Procesados 500 fotogramas
Procesados 600 fotogramas
Procesados 700 fotogramas
Procesados 800 fotogramas
Procesados 900 fotogramas
Procesados 1000 fotogramas
Procesados 1100 fotogramas
Procesados 1200 fotogramas
Procesados 1300 fotogramas
Procesados 1400 fotogramas
Procesados 1500 fotogramas
Procesados 1600 fotogramas
Procesados 1700 fotogramas
Procesados 1800 fotogramas
Procesados 1900 fotogramas
Procesados 2000 fotogramas
Procesados 2100 fotogramas
Procesados 2200 fotogramas
Procesados 2300 fotogramas
Procesados 2400 fotogramas
Procesados 2500 fotogramas
Procesados 2600 fotogramas
Procesados 2700 fotogramas
Procesados 2800 fotogramas
Procesamiento finalizado
Vídeo: resultado_ocr.mp4
CSV: detecciones_ocr.csv

Matrículas reconocidas:
Vehículo ID 1: 3324HZC (confianza OCR: 0.91, lecturas: 5)
Vehículo ID 46: 5724GGP (confianza OCR: 0.73, lecturas: 1)
Vehículo ID 34: 3713LKC (confianza O

### SmolVLM

In [12]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_ID)

vlm = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                dtype=torch.bfloat16,
                                                _attn_implementation="flash_attention_2" if device == "cuda" else "eager").to(device)

print("SmolVLM cargado en:", device)

SmolVLM cargado en: cpu


In [ ]:

def read_plate_vlm(crop):

    if crop is None or crop.size == 0:
        return ""

    # Ampliar matrícula
    crop = cv2.resize(crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(rgb)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text","text": (
                            "Read the license plate. "
                            "Return only the exact characters. "
                            "If unreadable, return NONE. "
                        )
                }
            ]
        }
    ]

    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(device)

    with torch.inference_mode():

        generated = vlm.generate(**inputs, max_new_tokens=20, do_sample=False)

    text = processor.batch_decode(
        generated,
        skip_special_tokens=True
    )[0].strip().upper()

    text = re.sub(r"[^A-Z0-9]", "", text)

    if re.fullmatch(r"[0-9]{4}[A-Z]{3}", text):
        return text

    return ""


In [ ]:
VIDEO_PATH = "C0142.mp4"
OUTPUT_VIDEO = "resultado_ocr.mp4"
OUTPUT_CSV = "detecciones_ocr.csv"

model_general = YOLO("yolo11n.pt")
model_plates = YOLO("runs/matriculas_yolo11n_1280_min001/weights/best.pt")

# OCR
reader = easyocr.Reader(['es'])

# Personas, bicicletas y vehículos
CLASSES = [0, 1, 2, 3, 5, 7]

# Letras permitidas en matrículas españolas
ALLOWLIST = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# Almacena las lecturas de cada vehículo
ocr_history = defaultdict(list)

# Resultado OCR más fiable por vehículo
best_plates = {}

def read_plate_ocr(crop):

    if crop is None or crop.size == 0:
        return "", 0.0

    # Ampliar matrícula para facilitar OCR
    crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # Pasar a escala de grises
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    # Mejorar contraste
    gray = cv2.equalizeHist(gray)

    results = reader.readtext(
        gray,
        detail=1,
        paragraph=False,
        allowlist=ALLOWLIST
    )

    if not results:
        return "", 0.0

    # Seleccionar resultado con mayor confianza
    best = max(results, key=lambda x: x[2])

    text = best[1].replace(" ", "").upper()
    confidence = float(best[2])

    print("Matrícula detectada: ", text)

    # Validar matricula: 4 números + 3 letras
    if not re.fullmatch(r"[0-9]{4}[ABCDEFGHIJKLMNOPQRSTUVWXYZ]{3}", text):
        return "", confidence

    return text, confidence

def update_ocr(vehicle_id, text, confidence):

    if vehicle_id is None or not text:
        return

    # Evitar lecturas de muy baja confianza
    if confidence < 0.30:
        return

    ocr_history[vehicle_id].append((text, confidence))

    # Contabilizar lecturas
    counts = Counter(item[0] for item in ocr_history[vehicle_id])

    # Elegir la lectura más repetida
    selected_text, occurrences = counts.most_common(1)[0]

    # Confianza media de las lecturas seleccionadas
    selected_confidences = [conf for plate, conf in ocr_history[vehicle_id] if plate == selected_text]

    best_plates[vehicle_id] = {
        "text": selected_text,
        "confidence": float(np.mean(selected_confidences)),
        "occurrences": occurrences
    }

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# Cabecera csv
headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "identificador_tracking",
    "x1", "y1", "x2", "y2",
    "matricula_en_su_caso",
    "confianza_matricula",
    "mx1", "my1", "mx2", "my2",
    "texto_matricula",
    "confianza_ocr",
    "lecturas_coincidentes"
]

csv_file = open(
    OUTPUT_CSV,
    "w",
    newline="",
    encoding="utf-8-sig"
)

writer = csv.DictWriter(csv_file, fieldnames=headers)
writer.writeheader()

frame_number = 0

print("COMIENZA LA DETECCION")

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # Detección de personas y vehículos
    result_general = model_general.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=CLASSES,
        conf=0.5,
        verbose=False
    )[0]

    # Detección de matrículas
    result_plates = model_plates.predict(
        frame,
        conf=0.4,
        verbose=False
    )[0]

    plates = []

    if result_plates.boxes is not None:

        for box in result_plates.boxes:

            px1, py1, px2, py2 = map(int, box.xyxy[0].tolist())

            px1 = max(0, px1)
            py1 = max(0, py1)
            px2 = min(width, px2)
            py2 = min(height, py2)

            if px2 <= px1 or py2 <= py1:
                continue

            plate_conf = float(box.conf[0])

            plate_crop = frame[py1:py2, px1:px2]

            # OCR
            plate_text, ocr_conf = read_plate_ocr(plate_crop)

            plates.append({
                "x1": px1,
                "y1": py1,
                "x2": px2,
                "y2": py2,
                "conf": plate_conf,
                "text": plate_text,
                "ocr_conf": ocr_conf
            })

    boxes = result_general.boxes

    if boxes is not None:

        for i, box in enumerate(boxes):

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            confidence = float(box.conf[0])
            class_id = int(box.cls[0])

            object_type = model_general.names[class_id]

            tracking_id = None

            if boxes.id is not None:
                tracking_id = int(boxes.id[i])

            matched_plate = None

            # Asocia matrículas a vehículos
            if class_id in [2, 3, 5, 7]:

                candidates = []

                for plate in plates:

                    cx = (plate["x1"] + plate["x2"]) / 2
                    cy = (plate["y1"] + plate["y2"]) / 2

                    if (x1 <= cx <= x2 and y1 <= cy <= y2):
                        # Priorizar el vehículo más pequeño
                        candidates.append(plate)

                if candidates:
                    matched_plate = min(candidates, key=lambda p: ((x2 - x1) * (y2 - y1)))

            if matched_plate and tracking_id is not None:

                update_ocr(
                    tracking_id,
                    matched_plate["text"],
                    matched_plate["ocr_conf"]
                )

            # Matrícula más fiable acumulada
            plate_info = best_plates.get(
                tracking_id, {}
            ) if tracking_id is not None else {}

            final_text = plate_info.get("text", "")
            final_ocr_conf = plate_info.get("confidence", "")
            occurrences = plate_info.get("occurrences", 0)

            # Guarda frame en csv
            row = {
                "fotograma": frame_number,
                "tipo_objeto": object_type,
                "confianza": confidence,
                "identificador_tracking": tracking_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "matricula_en_su_caso": int(
                    matched_plate is not None
                ),
                "confianza_matricula": (
                    matched_plate["conf"] if matched_plate else ""
                ),
                "mx1": matched_plate["x1"] if matched_plate else "",
                "my1": matched_plate["y1"] if matched_plate else "",
                "mx2": matched_plate["x2"] if matched_plate else "",
                "my2": matched_plate["y2"] if matched_plate else "",
                "texto_matricula": final_text,
                "confianza_ocr": final_ocr_conf,
                "lecturas_coincidentes": occurrences
            }

            writer.writerow(row)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            label = f"{object_type} ID:{tracking_id}"

            if final_text:
                label += f" | {final_text}"

            cv2.putText(frame, label, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    for plate in plates:

        cv2.rectangle(frame, (plate["x1"], plate["y1"]), (plate["x2"], plate["y2"]), (0, 0, 255), 2)

        label = plate["text"] if plate["text"] else "Matricula"

        cv2.putText(frame, label, (plate["x1"], max(20, plate["y1"] - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Guarda el vídeo
    out.write(frame)

    if frame_number % 100 == 0:
        print(f"Procesados {frame_number} fotogramas")

cap.release()
out.release()
csv_file.close()

print("Procesamiento finalizado")
print("Vídeo:", OUTPUT_VIDEO)
print("CSV:", OUTPUT_CSV)

print("\nMatrículas reconocidas:")

for vehicle_id, info in best_plates.items():

    print(
        f"Vehículo ID {vehicle_id}: "
        f"{info['text']} "
        f"(confianza OCR: {info['confidence']:.2f}, "
        f"lecturas: {info['occurrences']})"
    )
